# 🎒 Schulbedarf NLP Product Matcher, Subject & Level Training Pipeline

Dieses Notebook dient zum Trainieren und Evaluieren der Klassifikationsmodelle:
1. **Product Matcher Model:** Klassifiziert Textzeilen auf konkrete Produkt-IDs.
2. **Subject Classifier Model:** Sagt das Schulfach für eine Textzeile voraus.
3. **Level Classifier Model:** Sagt die Klassenstufe/Klasse für eine Textzeile voraus.

In [ ]:
import pandas as pd
import pickle
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

## 1. Mappings & Katalog-Daten laden und mergen

In [ ]:
mapped_path = "data/pdf_lines_mapped.csv"
products_path = "data/products.csv"

if not os.path.exists(mapped_path):
    raise FileNotFoundError(f"Mappings-Datei nicht gefunden unter: {mapped_path}")

df_mapped = pd.read_csv(mapped_path)
df_products = pd.read_csv(products_path)

# Merge mappings with catalog products to get subject and level labels
df_train = df_mapped.merge(df_products, left_on="product_id", right_on="id", how="inner")
print(f"Erfolgreich geladen: {len(df_train)} Trainingseinträge nach Merge.")
df_train[['raw_line', 'product_id', 'subject', 'level']].head(10)

## 2. Trainingsdaten vorbereiten

In [ ]:
df_train['raw_line'] = df_train['raw_line'].fillna("")
df_train = df_train[df_train['raw_line'].str.strip() != ""]

X = df_train['raw_line'].values
y_prod = df_train['product_id'].values
y_subj = df_train['subject'].values
y_lvl = df_train['level'].values

print(f"Training mit {len(X)} Zeilen.")

## 3. Modelle/Pipelines trainieren
Wir verwenden separate TF-IDF + LogisticRegression Pipelines. Ein getrenntes Training für Subject und Level ist deutlich robuster, da sich die relevanten Wort-Features stark unterscheiden.

In [ ]:
# Helper function to create and fit a pipeline
def train_pipeline(X_data, y_labels):
    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 6),
        min_df=1,
        sublinear_tf=True
    )
    classifier = LogisticRegression(
        C=20.0,
        class_weight='balanced',
        max_iter=2000,
        random_state=42
    )
    pipe = make_pipeline(vectorizer, classifier)
    pipe.fit(X_data, y_labels)
    return pipe

print("Training Product Matcher Model...")
pipeline_prod = train_pipeline(X, y_prod)

print("Training Subject Classifier Model...")
pipeline_subj = train_pipeline(X, y_subj)

print("Training Level Classifier Model...")
pipeline_lvl = train_pipeline(X, y_lvl)

## 4. Evaluierung

In [ ]:
print(f"Product matching Train Accuracy: {pipeline_prod.score(X, y_prod) * 100:.2f}%")
print(f"Subject classification Train Accuracy: {pipeline_subj.score(X, y_subj) * 100:.2f}%")
print(f"Level classification Train Accuracy: {pipeline_lvl.score(X, y_lvl) * 100:.2f}%")

## 5. Export der gebündelten Modelle

In [ ]:
model_path = "src/model.pkl"
model_data = {
    "pipeline": pipeline_prod,  # Abwärtskompatibel für bisherige Nutzung
    "product_pipeline": pipeline_prod,
    "subject_pipeline": pipeline_subj,
    "level_pipeline": pipeline_lvl
}

with open(model_path, "wb") as f:
    pickle.dump(model_data, f)

print(f"Alle 3 Modelle erfolgreich gebündelt gespeichert unter: {model_path}")